In [171]:
!pip install great_expectations -q
import great_expectations as gx
print(gx.__version__)

1.18.1


# Part B — Great Expectations Quality Gate

Implement a GX checkpoint with five expectations. Write a regulatory compliance comment above each — frame it as a risk control, not a tech check.

5.	loan_id must not be null and must be unique — duplicate IDs would allow the same application to be approved twice.
6.	credit_score must not be null — RBI guidelines require a bureau check before approval; a missing score means the check was skipped.
7.	credit_score must be between 300 and 900 — values outside this range are not valid bureau scores.
8.	annual_income must be greater than 0 — zero or negative income records indicate a data entry failure and cannot be used for affordability assessment.
9.	loan_status must be one of: Approved, Rejected, Defaulted, Closed — any other value indicates a system integration error.

Run the checkpoint, save Data Docs as HTML. Write a 3-bullet compliance memo for the CFO: which checks passed, which failed, and what the bank's exposure would be if the failing records were submitted for regulatory review


In [172]:
import pandas as pd
from google.colab import drive
from IPython.display import display, HTML
import os
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/FDE Projects/FinSight/finsight_loan_applications.csv')
print(df.shape)
print(df.columns.tolist())
df.head(3)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(130, 13)
['loan_id', 'application_date', 'applicant_age', 'employment_status', 'annual_income', 'loan_amount', 'loan_tenure_months', 'credit_score', 'existing_emis', 'loan_purpose', 'branch', 'loan_status', 'default_flag']


,loan_id,application_date,applicant_age,employment_status,annual_income,loan_amount,loan_tenure_months,credit_score,existing_emis,loan_purpose,branch,loan_status,default_flag
0,LN00001,2024-01-23,47,Retired,NaN,382358,48,498.0,4,Business,Hyderabad,Defaulted,1
1,LN00002,2024-09-17,48,Unemployed,NaN,1285279,60,782.0,4,Business,Hyderabad,Closed,0
2,LN00003,2024-06-22,36,Retired,NaN,1158273,36,521.0,4,Home Renovation,Delhi,Rejected,0


In [173]:
context = gx.get_context(mode="file", project_root_dir="/content")

In [174]:
import great_expectations as gx

# Get the context to manage GX artifacts
context = gx.get_context()

# Attempt to delete each artifact if it exists
# Use individual try-except blocks to handle DataContextError when an artifact is not found
try:
    context.checkpoints.delete("finsight_checkpoint")
    print("Checkpoint 'finsight_checkpoint' deleted")
except :
    print("Checkpoint 'finsight_checkpoint' not found, skipping deletion.")
try:
    context.validation_definitions.delete("finsight_validation")
    print("Validation Definition 'finsight_validation' deleted")
except :
    print("Validation Definition 'finsight_validation' not found, skipping deletion.")

try:
    context.data_sources.delete("finsight_datasources")
    print("Datasource 'finsight_datasource' deleted")
except gx.exceptions.DataContextError:
    print("Datasource 'finsight_datasource' not found, skipping deletion.")

try:
    context.suites.delete("finsight_risk_controls")
    print("Suite 'finsight_risk_controls' deleted")
except gx.exceptions.DataContextError:
    print("Suite 'finsight_risk_controls' not found, skipping deletion.")





print("Cleanup attempt completed.")

Datasource 'finsight_datasource' deleted
Suite 'finsight_risk_controls' deleted
Cleanup attempt completed.


In [175]:
import great_expectations as gx
import pandas as pd

context = gx.get_context()

datasource = context.data_sources.add_pandas("finsight_datasources")
data_asset = datasource.add_dataframe_asset(name="loan_applications")
batch_definition = data_asset.add_batch_definition_whole_dataframe("full_batch")
batch = batch_definition.get_batch(batch_parameters={"dataframe": df})

print("Batch loaded successfully")
print(f"Rows in batch: {len(df)}")

Batch loaded successfully
Rows in batch: 130


In [176]:

suite = context.suites.add(gx.ExpectationSuite(name="finsight_risk_controls"))

# REGULATORY CONTROL 1: Duplicate loan IDs would allow the same application
# to be approved twice, creating fraudulent double-disbursement risk.
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeUnique(column="loan_id"))
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="loan_id"))

# REGULATORY CONTROL 2: RBI guidelines require a bureau credit check before
# approval. A missing credit_score means the check was skipped entirely.
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="credit_score"))

# REGULATORY CONTROL 3: Values outside 300–900 are not valid bureau scores
# and indicate a system integration or data entry failure.
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
    column="credit_score", min_value=300, max_value=900))

# REGULATORY CONTROL 4: Zero or negative income records indicate a data entry
# failure and cannot be used for affordability assessment under RBI norms.
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
    column="annual_income", min_value=1))
# REGULATORY CONTROL 4b: Null annual_income cannot be used for affordability
# assessment — missing income data must be flagged before credit decisioning.
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="annual_income"))
print("Null check for annual_income added")

# REGULATORY CONTROL 5: Any loan_status value outside these four indicates
# a system integration error and must not enter the credit decisioning pipeline.
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeInSet(
    column="loan_status",
    value_set=["Approved", "Rejected", "Defaulted", "Closed"]))

print("6 expectations added successfully")

Null check for annual_income added
6 expectations added successfully


Checkpoint 'finsight_checkpoint' deleted
Validation Definition 'finsight_validation' deleted


In [178]:
validation_definition = context.validation_definitions.add(
    gx.ValidationDefinition(
        name="finsight_validation",
        data=batch_definition,
        suite=suite
    )
)

checkpoint = context.checkpoints.add(
    gx.Checkpoint(
        name="finsight_checkpoint",
        validation_definitions=[validation_definition],
    )
)

results = checkpoint.run(batch_parameters={"dataframe": df})

# Print summary
print(f"Overall success: {results.success}")
print(f"Overall success: {results.success}")
print("\n--- Individual Results ---")

for run_result in results.run_results.values():
    for result in run_result.results:
        expectation = result.expectation_config.type
        column = result.expectation_config.kwargs.get("column", "")
        status = "PASSED" if result.success else "FAILED"
        print(f"{status} | {column} | {expectation}")



Calculating Metrics:   0%|          | 0/43 [00:00<?, ?it/s]

Overall success: False
Overall success: False

--- Individual Results ---
PASSED | loan_id | expect_column_values_to_be_unique
PASSED | loan_id | expect_column_values_to_not_be_null
FAILED | credit_score | expect_column_values_to_not_be_null
PASSED | credit_score | expect_column_values_to_be_between
PASSED | annual_income | expect_column_values_to_be_between
FAILED | annual_income | expect_column_values_to_not_be_null
PASSED | loan_status | expect_column_values_to_be_in_set


In [179]:
both_missing = df[df['annual_income'].isnull() & df['credit_score'].isnull()]
print(f"Records missing BOTH: {len(both_missing)}")
print(both_missing[['loan_id', 'employment_status', 'annual_income', 'credit_score']])

Records missing BOTH: 0
Empty DataFrame
Columns: [loan_id, employment_status, annual_income, credit_score]
Index: []


## Part B — Great Expectations Compliance Results

### Checkpoint Summary
- PASSED: loan_id uniqueness and null check — no duplicate or missing application IDs
- PASSED: credit_score range (300–900) — all present scores are valid bureau values
- PASSED: annual_income > 0 — no zero or negative income records
- PASSED: loan_status valid values — no system integration errors detected
- FAILED: credit_score not null — 6 applications have no bureau score on record
- FAILED: annual_income not null — 5 applications have no income data on record

### CFO Compliance Memo

**To:** CFO, FinSight Risk Committee  
**Re:** Data Quality Gate Results — Loan Application Portfolio

1. **What passed:** Application ID integrity is clean — no duplicates or missing IDs.
   All recorded credit scores fall within valid RBI bureau range.
   No invalid loan status values were detected in the pipeline.

2. **What failed:** 6 applications (4.6%) have no credit score — meaning the RBI-mandated
   bureau check was skipped before approval decisions were made.
   5 applications (3.8%) have no income data — affordability assessment
   was conducted without a verified income figure.

3. **Regulatory exposure:** If these 11 records were submitted during an RBI audit,
   the bank would be unable to demonstrate compliance with mandatory pre-approval
   checks — creating direct exposure to supervisory action and potential penalties.

In [180]:
context.build_data_docs()

# Get the local path of the generated report
docs_sites = context.get_docs_sites_urls()
print(docs_sites)

[{'site_name': 'local_site', 'site_url': 'file:///content/gx/uncommitted/data_docs/local_site/index.html'}]


In [181]:
context.build_data_docs()

# Find and display the validation report inline
docs_root = "gx/uncommitted/data_docs/local_site/validations"
html_file = None

for root, dirs, filenames in os.walk(docs_root):
    for f in filenames:
        if f.endswith(".html"):
            html_file = os.path.join(root, f)

if html_file:
    with open(html_file, "r") as f:
        html_content = f.read()
    print(f"📄 Showing: {html_file}\n")
    display(HTML(html_content))
else:
    print("⚠️  Data Docs not found. Make sure Cell 6 ran successfully.")

📄 Showing: gx/uncommitted/data_docs/local_site/validations/finsight_risk_controls/__none__/20260622T125433.862713Z/finsight_datasources-loan_applications.html



,
Evaluated Expectations,7
Successful Expectations,5
Unsuccessful Expectations,2
Success Percent,≈71.43%
,
Great Expectations Version,1.18.1
Run Name,__none__
Run Time,2026-06-22T12:54:33Z
,
ge_load_time,20260622T125433.875005Z


In [182]:
from google.colab import files

docs_root = "gx/uncommitted/data_docs/local_site/validations"
html_file = None

for root, dirs, filenames in os.walk(docs_root):
    for f in filenames:
        if f.endswith(".html"):
            html_file = os.path.join(root, f)

if html_file:
    files.download(html_file)
    print("✅ Data Docs report downloaded — send this to the client before 10 AM")
else:
    print("⚠️  No report found. Run Cell 7 first.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Data Docs report downloaded — send this to the client before 10 AM
